In [22]:
!pip install gensim nltk pandas

import pandas as pd
import nltk
import string
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

# Download required NLTK resources
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('wordnet')




[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [23]:

# Download required NLTK resources
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [24]:
# Step 1: Load CSV file
df = pd.read_csv("K8 Reviews v0.2.csv")

In [5]:
# Step 2: Normalize text case
reviews = df['review'].astype(str).str.lower().tolist()

In [10]:
# Download required NLTK resources
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('wordnet')
# Add this line to download the missing resource
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [8]:
# Step 3: Tokenize reviews
tokenized_reviews = [word_tokenize(review) for review in reviews]

In [11]:
# Step 4: POS tagging
tagged_reviews = [pos_tag(tokens) for tokens in tokenized_reviews]

In [12]:
# Step 5: Extract only nouns
noun_tags = {'NN', 'NNS', 'NNP', 'NNPS'}
noun_only = [[word for word, tag in review if tag in noun_tags] for review in tagged_reviews]


In [13]:
# Step 6: Lemmatization
lemmatizer = WordNetLemmatizer()
lemmatized = [[lemmatizer.lemmatize(word) for word in review] for review in noun_only]


In [14]:
# Step 7: Remove stopwords and punctuation
stop_words = set(stopwords.words('english'))
punct = set(string.punctuation)
cleaned_reviews = [
    [word for word in doc if word not in stop_words and word not in punct and word.isalpha()]
    for doc in lemmatized
]

In [15]:
# Step 8: Topic Modeling with 12 topics
dictionary = corpora.Dictionary(cleaned_reviews)
corpus = [dictionary.doc2bow(text) for text in cleaned_reviews]


In [16]:
lda_model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=12, random_state=42, passes=10)

In [17]:
# Print top terms per topic
print("\nTop 10 terms per topic (12-topic model):")
for i, topic in lda_model.show_topics(formatted=True, num_topics=12, num_words=10):
    print(f"Topic {i}: {topic}")


Top 10 terms per topic (12-topic model):
Topic 0: 0.246*"issue" + 0.095*"network" + 0.045*"sim" + 0.026*"heating" + 0.025*"lot" + 0.023*"jio" + 0.018*"problem" + 0.018*"signal" + 0.016*"time" + 0.015*"game"
Topic 1: 0.368*"battery" + 0.090*"backup" + 0.070*"camera" + 0.052*"life" + 0.041*"performance" + 0.034*"drain" + 0.033*"device" + 0.015*"mark" + 0.014*"excellent" + 0.014*"mah"
Topic 2: 0.078*"speaker" + 0.071*"time" + 0.070*"feature" + 0.052*"heat" + 0.034*"screen" + 0.025*"speed" + 0.025*"ram" + 0.023*"gb" + 0.023*"need" + 0.023*"side"
Topic 3: 0.331*"mobile" + 0.197*"price" + 0.078*"range" + 0.029*"smartphone" + 0.029*"feature" + 0.029*"buy" + 0.023*"specification" + 0.018*"bit" + 0.014*"cell" + 0.008*"budget"
Topic 4: 0.166*"camera" + 0.082*"quality" + 0.078*"phone" + 0.019*"mode" + 0.018*"display" + 0.018*"performance" + 0.015*"feature" + 0.014*"music" + 0.013*"sound" + 0.012*"screen"
Topic 5: 0.113*"month" + 0.110*"performance" + 0.109*"charger" + 0.048*"box" + 0.038*"earpho

In [18]:
# Step 8.2: Coherence score
coherence_model = CoherenceModel(model=lda_model, texts=cleaned_reviews, dictionary=dictionary, coherence='c_v')
coherence_score = coherence_model.get_coherence()
print(f"\nCoherence Score (12-topic model): {coherence_score:.4f}")


Coherence Score (12-topic model): 0.5502


In [19]:
# Step 9 & 10: Try different numbers of topics to find the optimal one
def compute_coherence_values(dictionary, corpus, texts, start, limit, step):
    scores = []
    for num_topics in range(start, limit, step):
        model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics, random_state=42, passes=10)
        coherencemodel = CoherenceModel(model=model, texts=texts, dictionary=dictionary, coherence='c_v')
        scores.append((num_topics, coherencemodel.get_coherence()))
    return scores

print("\nFinding optimal number of topics...")
topic_range = compute_coherence_values(dictionary, corpus, cleaned_reviews, start=5, limit=20, step=1)
for num, score in topic_range:
    print(f"Topics: {num}, Coherence Score: {score:.4f}")


Finding optimal number of topics...
Topics: 5, Coherence Score: 0.5668
Topics: 6, Coherence Score: 0.5516
Topics: 7, Coherence Score: 0.5366
Topics: 8, Coherence Score: 0.5619
Topics: 9, Coherence Score: 0.5122
Topics: 10, Coherence Score: 0.5482
Topics: 11, Coherence Score: 0.5593
Topics: 12, Coherence Score: 0.5502
Topics: 13, Coherence Score: 0.5234
Topics: 14, Coherence Score: 0.5132
Topics: 15, Coherence Score: 0.5467
Topics: 16, Coherence Score: 0.5023
Topics: 17, Coherence Score: 0.4554
Topics: 18, Coherence Score: 0.4676
Topics: 19, Coherence Score: 0.5068


In [20]:
# Choose best topic number based on coherence score (manually or automatically)
best_num_topics = max(topic_range, key=lambda x: x[1])[0]
final_model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=best_num_topics, random_state=42, passes=10)


In [21]:
# Step 11: Name the topics and create a summary table
topic_summary = {}
for i, topic in final_model.show_topics(num_topics=best_num_topics, num_words=10, formatted=False):
    keywords = [word for word, _ in topic]
    topic_summary[f"Topic {i}"] = keywords

# Display as table
print("\nNamed Topics Summary:")
for topic, keywords in topic_summary.items():
    print(f"{topic}: {', '.join(keywords)}")



Named Topics Summary:
Topic 0: phone, issue, problem, amazon, service, network, lenovo, time, heating, month
Topic 1: battery, camera, performance, backup, quality, problem, hour, life, day, mobile
Topic 2: product, money, waste, delivery, value, h, worth, headphone, thanks, expectation
Topic 3: mobile, price, phone, range, hai, feature, ho, budget, smartphone, specification
Topic 4: phone, camera, note, quality, feature, screen, lenovo, call, sound, display
